In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

In [0]:

regex_demo = spark.createDataFrame([
    ("ABC123",),
    ("ABC456",),
    ("XYZ999",),
    ("ABC",),
    ("  ABC123  ",),
    ("A-B-C-123",)
], ["value"])
regex_demo.display()

In [0]:
from pyspark.sql.functions import regexp_replace
result = regex_demo.withColumn("remove_digits", regexp_replace("value", r"\d"," "))\
    .withColumn("remove-hyphes", regexp_replace("remove_digits", r"-"," "))\
        .withColumn("trim_value", regexp_replace("remove-hyphes", r"\s+",""))\
            .withColumn("trim_value", regexp_replace("value", r"\s+",""))
result.display()

In [0]:
from pyspark.sql.functions import regexp_extract,regexp_replace
result_1 = regex_demo.withColumn("remove-hyphes", regexp_replace("value", r"-"," "))\
                        .withColumn("trim_value", regexp_replace("remove-hyphes", r"\s+",""))\
                        .withColumn("letters", regexp_extract("trim_value", r"([A-Za-z]+)", 0))\
                            .withColumn("digits", regexp_extract("value", r"(\d+)",1)) # extracting only digits from the value column
result_1.display()

In [0]:
# regexp_replace --> modify or clean the string  from  the col
# where as regexp_extract --> extract the string from the column

In [0]:
# full load --> 1 files -->20 records..table lo load 
#            --> 2 files --> 30 records --> table lo load --> overwrite the table 
# incremental load --> 1 file --> 20 records --> table lo load --> append the table
#            --> 2 files --> 30 records --> table lo load --> merge condition sql based --> unique -- >  10 records --> table lo load update 10 records --> 20 records insert chesthundi 
#            30 --> 10 update 20 insert 20 + 20 ==>40 records 
#         1 . 20+30 ==>50 records 

In [0]:
# 1 file --> dubai --> watermark -- > 20 records --> 09-09-
# 2 file --> dubai --> watermark -->ingestion _timestamp col --> 10-->30[]
# 2 file --> germany --> watermark -->ingestion _timestamp col 

In [0]:
# merge into target table target
# using source table source 
# on target.id = source.id and 
#  target.c1 = source.c2

# when matched then update set *
# when not matched then insert

In [0]:
#basic aggregation in the pyspark count, sum, avg, min,max,distinct 
from pyspark.sql.functions import count,avg,min,max,sum
df=spark.table("formula1_dev.bronze.results")
df1=df.groupBy("driver_id").agg(count("*").alias("total_count"),
                                avg("points").alias("avg_points"),
                                min("points").alias("min_points"),
                                max("points").alias("max_points"),
                                sum("points").alias("total_points")
                                )
# df2=df.groupBy("driver_id").agg(avg("points").alias("avg_points"))
df1.display()

In [0]:
# UC names and tables
catalog = f"formula1_{env}"
bronze = "bronze"
silver = "silver"
spark.sql(f"create schema if not exists {catalog}.{silver}")
def bronze_table(name):
    return f"{catalog}.{bronze}.{name}"
def silver_table(name):
    return f"{catalog}.{silver}.{name}"

In [0]:
results=spark.table(bronze_table("results")) # bronze table name
print("Bronze rows:", results.count())
# display(results)
# results1=spark.table(f"formula1_dev.bronze.results")
# display(results1)

In [0]:
# target_name = spark.table(silver_table("results"))
# print("silver rows:", target_name.count()
from pyspark.sql.functions import col,lit
target_name = silver_table("results") # get the silver table name
if not spark.catalog.tableExists(target_name): # check if the silver table exists or not 
    print("silver target table does not exists") 
    results_batch = results # initial load 
else:
    last_ts = spark.table(target_name).agg(max("ingestion_timestamp").alias("max_ts")).first()["max_ts"] # get the last timestamp value from the silver table 
    print("last silver timestamp:", last_ts) # get the last timestamp value from the silver table
    results_batch=(
        results if last_ts is None # initial load 
        else results.filter(col("ingestion_timestamp")> lit(last_ts)) # incremental load
    )
print("rows selected:", results_batch.count())

In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists(target_name): # intial load
    (results_batch.write.mode("overwrite").format("delta").saveAsTable(target_name)) # writing into the target table 
else:
    target=DeltaTable.forName(spark,target_name) # incremental load 
    (target.alias("t")
     .merge(results_batch.alias("s"), "t.result_id = s.result_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute()
     )

In [0]:
%sql
select count(*) from formula1_dev.silver.results

In [0]:
# %sql
# merge into table target t
# using source s 
# on t.key_id  =  s.key_id and 
# t.c2 = s.c2
# when matched update set =
# when not matched insert values()